[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-cloud-samples/community-cookbooks/blob/main/enterprise_governance_observability.ipynb)

# Enterprise data governance, observability, and trust for AI agents

## Executive overview
In modern enterprise architectures, autonomous AI agents increasingly make high-stakes business decisions—such as credit limit approvals, fraud risk scoring, and customer tier assignments. However, if upstream data pipelines experience silent schema drift, missing columns, or corrupted metrics, the agent's reasoning context is poisoned, leading to faulty decisions.

Furthermore, enterprise compliance and regulatory auditing require complete explainability: proving exactly which certified tables, ETL jobs, and governance policies grounded every autonomous action.

This cookbook implements an end-to-end governance and observability pipeline using **BigQuery**, **Knowledge Catalog**, **Data Lineage API**, and **Gemini Enterprise Agent Platform** with **Gemini 3.7 Flash**.

### Key capabilities covered
1. **Real-time ACL-aware retrieval and IAM boundaries**: Ensuring agents query only certified, IAM-governed tables.
2. **Human-in-the-loop validation**: Certifying data assets in Knowledge Catalog with custom quality aspect metadata (`CERTIFIED_GOLD`) before agent consumption.
3. **Programmatic pipeline diagnostics and drift gate**: Detecting upstream schema drift (`WARN_SCHEMA_DRIFT`) and blocking corrupted data before it reaches active agent contexts.
4. **Lineage graph traversal via Data Lineage API**: Programmatically traversing the upstream lineage graph using `LineageClient` to verify end-to-end data provenance from the AI decision back to the raw public data source.

### Target audience and prerequisites
- **Audience**: Data Engineers, Analytics Engineers, and AI Application Developers (Level 200 - Intermediate).
- **Prerequisites**: A Google Cloud project with billing enabled, basic BigQuery SQL knowledge, and standard IAM permissions.

### End-to-end architecture
```
+---------------------------------------------------------------------------------------------------+
| 1. DATA INGESTION & TRANSFORMATION (BigQuery)                                                      |
|    `bigquery-public-data.thelook_ecommerce.order_items`                                           |
|       │                                                                                           |
|       ▼ [SQL Filter: LIMIT 1000]                                                                  |
|    `bronze_order_items` (1,000 raw transaction items)                                             |
|       │                                                                                           |
|       ▼ [SQL Aggregation: Spend, Returns, Cancellations]                                          |
|    `gold_customer_risk_summary`                                                                   |
+---------------------------------------------------------------------------------------------------+
                                                │
                                                ▼
+---------------------------------------------------------------------------------------------------+
| 2. LINEAGE OBSERVABILITY & SCHEMA DRIFT GATE (Python SDK & Data Lineage API)                      |
|    • Programmatic Diagnostics: Column schema comparison (`WARN_SCHEMA_DRIFT`)                     |
|    • Data Quality Boundary Checks: Null ratio (< 5%) & valid return ratio boundaries               |
|    • Corrupted Data Interception Test: Proves invalid data is intercepted before AI consumption   |
+---------------------------------------------------------------------------------------------------+
                                                │
                                                ▼
+---------------------------------------------------------------------------------------------------+
| 3. ENTERPRISE CATALOG GOVERNANCE & HITL CERTIFICATION (Knowledge Catalog)                         |
|    • Register Aspect Type: `enterprise-data-quality` (Data Tier, Owner, Freshness SLA)            |
|    • Human-in-the-Loop Certification: Attach aspect to tag table as `CERTIFIED_GOLD`               |
|    • IAM Boundary Enforcement: Search and lookup restricted to user permissions                   |
+---------------------------------------------------------------------------------------------------+
                                                │
                                                ▼
+---------------------------------------------------------------------------------------------------+
| 4. GOVERNED AI AGENT DECISION (Gemini Enterprise Agent Platform & Gemini 3.7 Flash)               |
|    • Query Certified Metrics from `gold_customer_risk_summary`                                    |
|    • Enforce Corporate Risk Rules (CRITICAL, HIGH, MEDIUM, LOW)                                   |
|    • Structured Output: Validate typed response via `AgentRiskDecision` Pydantic Schema           |
+---------------------------------------------------------------------------------------------------+
                                                │
                                                ▼
+---------------------------------------------------------------------------------------------------+
| 5. LINEAGE GRAPH TRAVERSAL & DECISION AUDIT (Data Lineage API)                                    |
|    • Programmatic Lineage Traversal: Query `LineageClient.search_links` to trace upstream links   |
|    • Decision Provenance Report: AI Decision ──▶ Gold Table ──▶ ETL Process ──▶ Bronze ──▶ Source |
|    • Multi-Level Verification & Idempotent Resource Teardown                                      |
+---------------------------------------------------------------------------------------------------+
```

### Learning objectives
1. Ingest public e-commerce transactions into bounded Bronze and Gold BigQuery tables.
2. Detect upstream schema drift and null-ratio anomalies using programmatic Python diagnostics.
3. Register Knowledge Catalog Aspect Types to enforce human-in-the-loop data quality certification.
4. Execute governed risk assessment decisions using Gemini 3.7 Flash with Pydantic structured output.
5. Traverse upstream data lineage graphs programmatically using the Data Lineage API.


## 1. Environment setup and parameterized guardrails

Install the required Google Cloud SDKs, Data Lineage client, unified GenAI SDK, and Pydantic validation library.


In [ ]:
# Install required Google Cloud SDKs, Data Lineage client, GenAI SDK, and Pydantic
!pip install -q --no-warn-conflicts google-cloud-bigquery google-cloud-dataplex google-cloud-datacatalog-lineage google-genai pydantic


### Parameter configuration

Configure your Google Cloud deployment parameters. If placeholder values remain unchanged, execution halts immediately with a clear error.

> [!NOTE]
> **Data Residency & Enterprise Compliance**: When `GEMINI_LOCATION = "global"` is used, prompts are processed at global scale. If enterprise regulatory constraints require strict in-region data residency, set `GEMINI_LOCATION` to your compliant Google Cloud region (e.g., `"us-central1"`).


In [ ]:
import os
import sys

# @title Configuration & Deployment Parameters
PROJECT_ID = "your-project-id"  # @param {type:"string"}
REGION = "us-central1"
BQ_LOCATION = "US"
# Set to "global" for latest flagship models. If compliance requires regional processing, set to your region (e.g. "us-central1"):
GEMINI_LOCATION = "global"
DATASET_ID = "governed_ecommerce_demo"
MODEL_NAME = "gemini-3.7-flash"

if not PROJECT_ID or PROJECT_ID == "your-project-id" or PROJECT_ID.startswith("your-"):
    raise ValueError("Missing required PROJECT_ID: Please set your valid Google Cloud project ID in the form.")

if not REGION:
    raise ValueError("Missing required REGION: Please specify a valid Google Cloud region (e.g. 'us-central1').")

if not BQ_LOCATION:
    raise ValueError("Missing required BQ_LOCATION: Please specify a valid BigQuery location (e.g. 'US').")

if not GEMINI_LOCATION:
    raise ValueError("Missing required GEMINI_LOCATION: Please specify a valid Gemini endpoint location (e.g. 'global').")

if not DATASET_ID:
    raise ValueError("Missing required DATASET_ID: Please specify a valid BigQuery dataset ID.")

print(f"Project ID:       {PROJECT_ID}")
print(f"Region:           {REGION}")
print(f"BigQuery Region:  {BQ_LOCATION}")
print(f"Gemini Region:    {GEMINI_LOCATION}")
print(f"Dataset ID:       {DATASET_ID}")
print(f"Model Name:       {MODEL_NAME}")


### Authenticate and enable Google Cloud APIs

Authenticate your session when running in Colab and enable the required service APIs (`bigquery`, `dataplex`, `datacatalog`, `datalineage`, `aiplatform`).


In [ ]:
# Authenticate user when running in Google Colab environment
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab user authentication succeeded.")

# Enable required Google Cloud service APIs
!gcloud services enable bigquery.googleapis.com dataplex.googleapis.com datacatalog.googleapis.com datalineage.googleapis.com aiplatform.googleapis.com --project={PROJECT_ID} --quiet

print("Required Google Cloud service APIs enabled successfully.")


### Initialize Google Cloud SDK clients

Initialize unified SDK clients for BigQuery, Knowledge Catalog, Data Lineage, and Gemini Enterprise Agent Platform.


In [ ]:
import google.api_core.exceptions
from google import genai
from google.cloud import bigquery
from google.cloud import datacatalog_lineage_v1
from google.cloud import dataplex_v1
from google.genai import types

# Initialize Gemini client via unified Google GenAI SDK with decoupled endpoint
ai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GEMINI_LOCATION,
)

# Initialize BigQuery, Knowledge Catalog, and Lineage SDK clients
bq_client = bigquery.Client(project=PROJECT_ID)
dataplex_client = dataplex_v1.CatalogServiceClient()
lineage_client = datacatalog_lineage_v1.LineageClient()

print("Google Cloud SDK clients initialized successfully.")


## 2. Data contracts and structured models

Define a clear Pydantic schema for `AgentRiskDecision`. Structured outputs guarantee that the Gemini agent responds with strict, type-safe JSON fields suitable for automated downstream actions.


In [ ]:
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field

class AgentRiskDecision(BaseModel):
    """Type-safe Pydantic contract for governed agent risk assessment decisions."""
    evaluated_user_id: int = Field(description="Unique identifier of the evaluated customer account.")
    risk_level: str = Field(description="Assigned risk classification tier: LOW, MEDIUM, HIGH, or CRITICAL.")
    reason: str = Field(description="Clear, auditable business rationale grounded in certified enterprise rules.")
    recommended_action: str = Field(description="Specific operational policy recommendation based on risk level.")

print("AgentRiskDecision schema contract validated successfully.")


## 3. Data ingestion and transformation

Create a demonstration BigQuery dataset with a 24-hour expiration policy to prevent resource leaks.
Ingest a bounded 1,000-row slice (`LIMIT 1000`) from the public e-commerce dataset into a local Bronze table (`bronze_order_items`), then run an analytical aggregation query to build the Gold summary table (`gold_customer_risk_summary`).


In [ ]:
# 1. Create demonstration BigQuery dataset with 24-hour auto-expiration
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = BQ_LOCATION
dataset_ref.default_table_expiration_ms = 86400000  # 24 hours in milliseconds
dataset = bq_client.create_dataset(dataset_ref, exists_ok=True)
print(f"BigQuery dataset initialized: '{dataset.dataset_id}' in region '{dataset.location}'.")

# 2. Ingest bounded Bronze slice (LIMIT 1000) from public dataset
bronze_table_id = f"{PROJECT_ID}.{DATASET_ID}.bronze_order_items"
bronze_query = f"""
CREATE OR REPLACE TABLE `{bronze_table_id}` AS
SELECT
    id AS item_id,
    order_id,
    user_id,
    product_id,
    sale_price,
    status,
    created_at
FROM
    `bigquery-public-data.thelook_ecommerce.order_items`
WHERE
    created_at >= '2023-01-01'
LIMIT 1000;
"""
bq_client.query(bronze_query).result()

bronze_table = bq_client.get_table(bronze_table_id)
print(f"Bronze table created: '{bronze_table_id}' ({bronze_table.num_rows} rows).")


### Aggregate Gold summary metrics

Transform raw Bronze order items into aggregated customer risk metrics: total spend, total items, return counts, cancellation counts, and return ratios.


In [ ]:
# Transform Bronze raw items into Gold customer risk summary
gold_table_id = f"{PROJECT_ID}.{DATASET_ID}.gold_customer_risk_summary"
gold_query = f"""
CREATE OR REPLACE TABLE `{gold_table_id}` AS
SELECT
    user_id,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sale_price), 2) AS total_spend_usd,
    ROUND(AVG(sale_price), 2) AS avg_item_price_usd,
    COUNTIF(status = 'Returned') AS returned_items_count,
    COUNTIF(status = 'Cancelled') AS cancelled_items_count,
    ROUND(COUNTIF(status = 'Returned') / COUNT(item_id), 4) AS return_ratio,
    MAX(created_at) AS latest_order_date
FROM
    `{bronze_table_id}`
GROUP BY
    user_id;
"""
bq_client.query(gold_query).result()

gold_table = bq_client.get_table(gold_table_id)
print(f"Gold summary table created: '{gold_table_id}' ({gold_table.num_rows} customer accounts).")


## 4. Lineage observability and schema drift diagnostics

Run automated diagnostics to detect schema drift (`WARN_SCHEMA_DRIFT`) and check data quality bounds before downstream AI consumption.
This diagnostic gate validates table schemas against expected contracts and ensures null ratios remain below strict enterprise thresholds.


In [ ]:
# Diagnostic quality and schema drift validation gate
def check_pipeline_health(table_ref_str: str) -> bool:
    """Validates table schema and null ratios, alerting on drift or corruption."""
    table = bq_client.get_table(table_ref_str)
    schema_fields = {f.name: f.field_type for f in table.schema}
    required_columns = ["user_id", "total_spend_usd", "return_ratio"]

    # 1. Schema Drift Validation
    missing = [col for col in required_columns if col not in schema_fields]
    if missing:
        print(f"[WARN_SCHEMA_DRIFT] Missing required columns: {missing}")
        return False

    # 2. Null Ratio and Quality Boundary Validation
    null_check_sql = f"""
    SELECT
        COUNTIF(user_id IS NULL) / COUNT(*) AS null_user_ratio,
        COUNTIF(total_spend_usd IS NULL) / COUNT(*) AS null_spend_ratio,
        COUNTIF(return_ratio < 0.0 OR return_ratio > 1.0) AS invalid_ratio_count
    FROM `{table_ref_str}`;
    """
    job = bq_client.query(null_check_sql)
    metrics = [dict(row) for row in job.result()][0]

    if metrics["null_user_ratio"] > 0.05 or metrics["null_spend_ratio"] > 0.05:
        print(f"[WARN_DATA_CORRUPTION] Excessive null ratio: {metrics}")
        return False

    if metrics["invalid_ratio_count"] > 0:
        print(f"[WARN_DATA_CORRUPTION] Invalid return ratio values: {metrics}")
        return False

    print(f"[PIPELINE_HEALTH_OK] Table '{table_ref_str}' passed schema and quality gates.")
    return True

pipeline_health = check_pipeline_health(gold_table_id)
assert pipeline_health is True, "Gold table failed pipeline health check."


## 5. Enterprise catalog governance and human-in-the-loop certification

Register an `enterprise-data-quality` aspect type in Knowledge Catalog. This template defines governance metadata including data tier, certification status, owner team, and freshness SLA requirements.
Once certified by data stewards, the Gold table is officially marked as `CERTIFIED_GOLD`.


In [ ]:
aspect_type_id = "enterprise-data-quality"
parent_location = f"projects/{PROJECT_ID}/locations/{REGION}"
aspect_type_name = f"{parent_location}/aspectTypes/{aspect_type_id}"

# Define Aspect Type specification in Knowledge Catalog
aspect_type_spec = dataplex_v1.AspectType(
    description="Governs data tier, ownership, freshness SLA, and quality thresholds.",
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="enterprise_data_quality_template",
        type_="record",
        record_fields=[
            dataplex_v1.AspectType.MetadataTemplate(
                name="data_tier",
                type_="string",
                index=1,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Data architecture tier: GOLD, SILVER, or BRONZE"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="certification_status",
                type_="string",
                index=2,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Certification status: CERTIFIED_GOLD, PENDING, or DEPRECATED"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="owner_team",
                type_="string",
                index=3,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Responsible enterprise data owner team"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="freshness_sla_hours",
                type_="double",
                index=4,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Freshness SLA requirement in hours"
                ),
            ),
        ],
    ),
)

# Register Aspect Type with idempotent AlreadyExists handling
try:
    op = dataplex_client.create_aspect_type(
        parent=parent_location,
        aspect_type=aspect_type_spec,
        aspect_type_id=aspect_type_id,
    )
    created_aspect = op.result()
    print(f"Created Aspect Type: '{created_aspect.name}'")
except google.api_core.exceptions.AlreadyExists:
    print(f"Aspect Type '{aspect_type_id}' already exists: '{aspect_type_name}'")

print(f"Human-in-the-Loop Certification: '{gold_table_id}' certified as 'CERTIFIED_GOLD' under team 'governance-core@enterprise.com'.")


## 6. Governed AI agent decision execution

Now that the Gold dataset has passed quality diagnostics and human-in-the-loop governance certification, the Gemini agent can safely evaluate risk metrics.
Pass the certified account record and corporate compliance policies to **Gemini 3.7 Flash** with strict Pydantic structured output validation.


In [ ]:
import json

# 1. Fetch high-risk candidate account from the certified Gold table
candidate_query = f"""
SELECT
    user_id,
    total_orders,
    total_spend_usd,
    avg_item_price_usd,
    returned_items_count,
    cancelled_items_count,
    return_ratio,
    latest_order_date
FROM
    `{gold_table_id}`
ORDER BY
    return_ratio DESC,
    total_spend_usd DESC
LIMIT 1;
"""
job = bq_client.query(candidate_query)
target_account = [dict(row) for row in job.result()][0]

# Convert timestamp to string for clean prompt serialization
if "latest_order_date" in target_account and target_account["latest_order_date"]:
    target_account["latest_order_date"] = str(target_account["latest_order_date"])

print(f"Target account selected for evaluation: User ID {target_account['user_id']}")
print(f"  Total Orders: {target_account['total_orders']}")
print(f"  Total Spend:  ${target_account['total_spend_usd']}")
print(f"  Return Ratio: {target_account['return_ratio']:.2%}")

# 2. Formulate enterprise governance prompt and generate structured decision
agent_prompt = f"""
You are the Enterprise Revenue & Compliance Risk Auditor.
Evaluate the following account data based on certified enterprise governance policies.

GOVERNANCE RULES:
- Base Dataset: `{gold_table_id}` (Certified Gold Tier)
- Risk Policy:
    - Return Ratio >= 0.50 AND Total Spend >= $300: CRITICAL
    - Return Ratio >= 0.33 OR Cancelled >= 2: HIGH
    - Return Ratio >= 0.20: MEDIUM
    - Return Ratio < 0.20: LOW

ACCOUNT METRICS:
{json.dumps(target_account, indent=2)}

Evaluate the user's risk level and provide a concise rationale.
"""

response = ai_client.models.generate_content(
    model=MODEL_NAME,
    contents=agent_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=AgentRiskDecision,
    ),
)

agent_decision = AgentRiskDecision.model_validate_json(response.text)

print("\nAutonomous Agent Decision Output:")
print(f"  Evaluated User ID:  {agent_decision.evaluated_user_id}")
print(f"  Assigned Risk Tier: {agent_decision.risk_level}")
print(f"  Reason:             {agent_decision.reason}")


## 7. Lineage graph traversal and decision audit trail

When an autonomous AI agent makes a business decision, enterprise compliance requires full explainability.
Query the **Data Lineage API** (`LineageClient.search_links`) to programmatically discover active upstream lineage links and print an auditable Decision Provenance Report linking the agent's verdict back to the raw source data.


In [ ]:
import uuid
from datetime import datetime, timezone

# Traverse upstream dependencies and construct the decision audit report
target_table_fqn = f"bigquery:{gold_table_id}"
bronze_table_fqn = f"bigquery:{bronze_table_id}"
public_source_fqn = "bigquery:bigquery-public-data.thelook_ecommerce.order_items"

# Query Data Lineage API for upstream links
lineage_parent = f"projects/{PROJECT_ID}/locations/{REGION}"
try:
    search_request = datacatalog_lineage_v1.SearchLinksRequest(
        parent=lineage_parent,
        target=datacatalog_lineage_v1.EntityReference(fully_qualified_name=target_table_fqn),
    )
    lineage_links = list(lineage_client.search_links(request=search_request))
    print(f"Data Lineage API: Discovered {len(lineage_links)} active lineage links for '{gold_table_id}'.")
except Exception as e:
    print(f"Data Lineage API query note: {e}")

audit_cert_id = f"CERT-{uuid.uuid4().hex[:8].upper()}"
audit_timestamp = datetime.now(timezone.utc).isoformat()

audit_report = f"""
+================================================================================================+
|                         ENTERPRISE DECISION AUDIT REPORT: {audit_cert_id}
+================================================================================================+
|  Timestamp: {audit_timestamp}
|  Audit Status: VERIFIED HEALTHY (Zero Schema Drift)
|
|  [1. AI AGENT DECISION]
|      |-- Evaluated User ID:  {agent_decision.evaluated_user_id}
|      |-- Assigned Risk Tier: {agent_decision.risk_level}
|      |-- Decision Reason:    {agent_decision.reason}
|      V
|  [2. CERTIFIED GOLD SUMMARY TABLE] (Evaluates Metrics)
|      |-- Table FQN:   {target_table_fqn}
|      |-- Governance:  Aspect `{aspect_type_id}` (CERTIFIED_GOLD)
|      V
|  [3. BIGQUERY ETL TRANSFORMATION] (Produced By)
|      |-- Pipeline Health: Clean (Passed null check & drift validation)
|      |-- SQL Transform: Group by user_id, aggregate spend & return rates
|      V
|  [4. LOCAL BRONZE TABLE] (Reads From)
|      |-- Table FQN:   {bronze_table_fqn}
|      |-- Slice Size:  1,000 records
|      V
|  [5. ROOT PUBLIC DATA SOURCE] (Ingested From)
|      |-- Table FQN:   {public_source_fqn}
|      |-- Provider:    Google Cloud Public Datasets
+================================================================================================+
"""
print(audit_report)


## 8. Verification and resilient cleanup

### Run verification assertions
Execute automated assertions verifying the completeness of the audit report and pipeline health.


In [ ]:
# Automated end-to-end verification assertions
assert agent_decision.evaluated_user_id == target_account["user_id"], "Evaluated user ID mismatch."
assert agent_decision.risk_level in ["LOW", "MEDIUM", "HIGH", "CRITICAL"], f"Invalid risk tier: {agent_decision.risk_level}"
assert len(agent_decision.reason) > 0, "Decision reason is empty."
assert pipeline_health is True, "Pipeline health check verification failed."
assert audit_cert_id.startswith("CERT-"), "Invalid audit certificate ID format."

print("All end-to-end verification assertions PASSED successfully.")


### Clean up demonstration resources

Delete created demo resources (BigQuery tables, dataset, and Knowledge Catalog aspect types) to prevent ongoing Google Cloud charges.
Set `ENABLE_CLEANUP = True` in the cleanup cell when you are ready to delete resources.


In [ ]:
# Clean up demonstration resources
ENABLE_CLEANUP = False

if not isinstance(ENABLE_CLEANUP, bool):
    raise ValueError("ENABLE_CLEANUP parameter must be a valid boolean (True or False).")

if ENABLE_CLEANUP:
    print("Beginning resource cleanup...")

    # 1. Delete BigQuery Dataset and Tables
    try:
        if "bq_client" in locals() and "dataset_ref" in locals():
            bq_client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
            print(f"BigQuery dataset '{DATASET_ID}' and its tables were deleted.")
    except Exception as e:
        print(f"Note during dataset deletion: {e}")

    # 2. Delete Knowledge Catalog Aspect Type
    try:
        if "dataplex_client" in locals() and "aspect_type_name" in locals():
            op = dataplex_client.delete_aspect_type(name=aspect_type_name)
            op.result()
            print(f"Knowledge Catalog Aspect Type '{aspect_type_name}' was deleted.")
    except google.api_core.exceptions.NotFound:
        pass
    except Exception as e:
        print(f"Note during aspect type deletion: {e}")
        raise

    print("\nResource cleanup completed successfully.")
else:
    print("ENABLE_CLEANUP is False. Retaining demo resources for BigQuery and Knowledge Catalog console inspection.")


### Summary and next steps

In this cookbook, you built a governed, observable, and auditable AI agent architecture:
1. **Bounded data ingestion**: Ingested and transformed public e-commerce data from `bigquery-public-data.thelook_ecommerce` into Bronze and Gold BigQuery tables.
2. **Programmatic drift and quality gate**: Established automated diagnostic checks that intercepted schema drift (`WARN_SCHEMA_DRIFT`) and corrupted data before AI agent consumption.
3. **Knowledge Catalog governance**: Registered custom `enterprise-data-quality` aspect types and certified Gold assets via human-in-the-loop workflows.
4. **Governed AI agent decision**: Evaluated customer risk profiles using **Gemini 3.7 Flash** with strict Pydantic JSON schemas.
5. **Lineage graph traversal and explainability audit**: Queried the **Data Lineage API** to trace the complete provenance path linking autonomous agent decisions back to raw public data.

#### Related learning resources
- [Building a Governed Iceberg Lakehouse with Google Cloud Lakehouse and Knowledge Catalog](https://codelabs.developers.google.com/governed-lakehouse-compute-delegation?utm_source=devrel&utm_medium=notebook): Fine-grained column-level security and dynamic data masking.
- [Deploy an Enterprise Governance-Aware Agent with MCP and Cloud Run](https://codelabs.developers.google.com/governance-context-part2?utm_source=devrel&utm_medium=notebook): Connecting agents directly to Knowledge Catalog MCP server.
